# 🧠 Week 6 Lab — Student Version
## From Muscles to Neurons: What Can Motor Cortex Tell Us That Muscles Cannot?

In Week 5 we classified healthy vs impaired subjects using EMG alone (85% accuracy, AUC = 0.94). This week we go **one level upstream** in the motor pathway — from muscles to the motor cortex neurons that drive them.

The lab follows the 7 sections of the lecture:

| Part | Lecture Section | What you will do |
|------|----------------|-----------------|
| 1 | §1–2: Going Upstream & Cosine Tuning | Explore the neural population and visualize tuning curves |
| 2 | §3: Modeling Spike Counts | See why linear/logistic regression fail; understand the Poisson GLM |
| 3 | §4: Encoding | Fit Poisson GLMs to recover each neuron's preferred direction |
| 4 | §5: Decoding (Population Vector) | Implement the parameter-free population vector decoder |
| 5 | §5: Decoding (Logistic Regression) | Compare neural vs EMG inputs with the same classifier |
| 6 | §6: The Clinical Question | Investigate why muscles beat neurons for diagnosis |
| 7 | §7: Bringing It Together | Update the running comparison table |

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.stats import poisson

from sklearn.linear_model import LogisticRegression, PoissonRegressor, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, roc_curve, auc, confusion_matrix

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
from google.colab import files
uploaded = files.upload()

---

## 🟢 Part 1: Going Upstream — Meet the Neural Population (Lecture §1–2)

In the lecture, we learned that every muscle activation pattern originates from neural commands in primary motor cortex (M1). Georgopoulos (1982) showed that each neuron has a **preferred direction (PD)** — the reach direction that makes it fire most vigorously — and that firing rates follow a **cosine tuning curve**:

```
firing rate = baseline + depth × cos(θ − θ_PD)
```

Our dataset now includes 80 simulated M1 neurons alongside the 6 EMG channels from Week 4.

In [ ]:
# Exercise 1.1: Load the Week 6 dataset
# Print all available fields and their shapes.
# Assign the key variables: X_raw, neural_rates, labels, targets, subjects,
# target_angles, neuron_pds, muscle_names
# Create boolean masks: healthy_mask, impaired_mask

### YOUR CODE HERE ###

### Exercise 1.2: Visualize cosine tuning

The lecture (Figure 2) showed that each neuron's firing rate traces a cosine as a function of reach direction. Reproduce this for **4 neurons** with different PDs (e.g., neurons 0, 20, 40, 60).

For each neuron, plot the **mean firing rate ± SEM** at each of the 8 directions, using **healthy subjects only**. Mark the true PD with a vertical dashed line.

**Expected:** Each neuron shows a smooth cosine shape peaking at its PD.

In [ ]:
# Exercise 1.2: Tuning curves for 4 example neurons (healthy only)
# For each neuron and each direction:
#   mask = (targets == d) & healthy_mask
#   mean = neural_rates[mask, neuron_idx].mean()
#   sem  = neural_rates[mask, neuron_idx].std() / sqrt(n)

### YOUR CODE HERE ###

### Exercise 1.2b: Healthy vs Impaired tuning (Lecture Figure 2c)

The lecture's Figure 2c shows **Neuron 21** with healthy (blue) and impaired (red) tuning curves overlaid. Reproduce this: plot mean ± SEM for both groups on the same axes.

**Expected:** The impaired curve has a lower peak and higher trough — reduced modulation depth. Error bars nearly overlap.

In [ ]:
# Exercise 1.2b: Neuron 21 healthy vs impaired tuning

### YOUR CODE HERE ###

### Exercise 1.3: Population activity heatmap

The lecture (Figure 3) showed that when we sort neurons by PD and plot average firing rates, a diagonal ridge appears — each direction activates neurons with matching PDs.

Create **two heatmaps** side by side: one for healthy, one for impaired subjects. Sort neurons (y-axis) by their true PD. Reach directions on x-axis.

**Expected:** Healthy shows a sharp diagonal ridge; impaired shows a blurred ridge (synergy merging propagates upstream).

In [ ]:
# Exercise 1.3: Population heatmaps — healthy vs impaired
# Sort neurons by PD: sort_idx = np.argsort(neuron_pds)
# Build a (80 x 8) matrix of mean rates, then reorder rows by sort_idx

### YOUR CODE HERE ###

---

## 🟢 Part 2: Why We Need a New Tool (Lecture §3)

The lecture showed that linear regression can predict **negative** firing rates (biologically impossible), and logistic regression caps at **1** (useless for rates of 5, 15, 30 spk/s). The Poisson GLM uses the **exponential link** to guarantee positive, unbounded outputs.

Let's see these problems firsthand.

### Exercise 2.1: Show that linear regression predicts negative rates (Lecture Figure 5b)

Construct a demonstration neuron using the **exponential cosine tuning** model:
```
true_rate = exp(β₀ + β₁·cos(θ − PD))
```
Use β₀=1.2, β₁=1.8, PD=90°. Generate Poisson spike counts from this neuron, then fit both `LinearRegression` and `PoissonRegressor`.

**Plot** all three curves: the true rate (always positive), linear regression (goes negative!), and Poisson GLM (stays positive). Shade the negative region.

**Key point:** The true rate is always positive, but linear regression has no such guarantee.

In [ ]:
# Exercise 2.1: Linear regression vs Poisson GLM
# 1. Set up: PD=90°, beta0=1.2, beta1=1.8
# 2. Generate true rates: exp(beta0 + beta1*cos(angles - PD))
# 3. Sample spike counts: np.random.poisson(true_rates)
# 4. Fit LinearRegression and PoissonRegressor on cos/sin features
# 5. Plot all three curves, shade where linear goes negative

### YOUR CODE HERE ###

### Exercise 2.2: Visualize the Poisson distribution

The lecture (Figure 5d) showed Poisson distributions at three firing rates. Recreate this: plot `poisson.pmf(k, λ)` for λ = 5, 12, and 25 on the same axes.

**Observe:** At low λ the distribution is skewed right (can't go below 0); at high λ it approaches a Gaussian. Variance equals the mean.

In [ ]:
# Exercise 2.2: Poisson distributions at three rates
# Use scipy.stats.poisson.pmf(k, lam) for k = 0, 1, 2, ..., 50

### YOUR CODE HERE ###

### Exercise 2.3: Likelihood and MLE

The lecture introduced MLE through a concrete example: you observe y = 18 spikes. For each possible λ, compute P(y=18 | λ) using the Poisson formula. Plot this **likelihood curve** and mark the peak.

**Expected:** The peak is at λ = 18 — the MLE is the observed count.

In [ ]:
# Exercise 2.3: Likelihood function for a single observation
# lambda_range = np.linspace(1, 40, 200)
# likelihood = poisson.pmf(18, lambda_range)
# Plot and mark the MLE peak

### YOUR CODE HERE ###

---

## 🟡 Part 3: Encoding — What Is Each Neuron Doing? (Lecture §4)

The lecture asked: given the reach direction, what is each neuron's role? We fit a **Poisson GLM** per neuron:

```
spike count ~ Poisson(exp(β₀ + β₁·cos(θ) + β₂·sin(θ)))
```

The cos/sin trick makes the model linear in parameters. We recover PD as `atan2(β₂, β₁)`.

**Important:** Fit on **healthy subjects only** — this establishes what each neuron *should* be doing.

In [ ]:
# Exercise 3.1: Fit Poisson GLM for all 80 neurons
# For each neuron:
#   1. Extract healthy trials: y = neural_rates[healthy_mask, ni]
#   2. Build X = [cos(angles), sin(angles)] where angles = target_angles[targets[healthy_mask]]
#   3. Fit PoissonRegressor(alpha=0, max_iter=1000)
#   4. Recover PD: np.arctan2(coef[1], coef[0]) % (2*pi)
#   5. Store recovered PD and depth = sqrt(coef[0]**2 + coef[1]**2)

### YOUR CODE HERE ###

### Exercise 3.2: Validate — recovered vs true PDs

The lecture (Figure 6) showed that recovered PDs cluster tightly along the diagonal. Create:
1. A **scatter plot** of recovered vs true PD (should lie on the diagonal)
2. A **histogram** of angular errors with the median marked

**Hint:** Angular error = `|angle(exp(j*(recovered - true)))|`, then convert to degrees.

**Expected:** Median error ~1–2° (the model is correctly specified).

In [ ]:
# Exercise 3.2: Recovered vs true PDs + error distribution

### YOUR CODE HERE ###

### Exercise 3.3: Visualize one neuron's encoding model (Lecture Figure 6c)

Pick neuron 20 and overlay:
- **Observed** mean rates ± **SD** at each direction (scatter with error bars)
- **Predicted** Poisson GLM curve over a fine grid (smooth line)
- Vertical lines for true PD and recovered PD

**Note:** Use SD (not SEM) for error bars to show trial-to-trial variability, matching the lecture figure.

In [ ]:
# Exercise 3.3: Single neuron encoding model visualization

### YOUR CODE HERE ###

---

## 🟡 Part 4: The Population Vector — Decoding Without ML (Lecture §5)

The lecture explained that the population vector **inverts the encoding model analytically** — no training needed. Each neuron votes for its PD, weighted by its firing rate:

```
PV = Σᵢ rᵢ · (cos θ_PD,i, sin θ_PD,i)
decoded direction = atan2(PV_y, PV_x)
```

This works because the encoding model (cosine tuning) already tells us the decoding rule. The lecture emphasized: **the Poisson GLM cannot decode** (it outputs firing rates, not direction labels), so we need a different approach.

In [ ]:
# Exercise 4.1: Implement the population vector decoder
def pop_vector_decode(rates, pds):
    """Decode reach direction using the population vector.
    Args:
        rates: (n_trials, n_neurons) firing rate matrix
        pds: (n_neurons,) preferred directions in radians
    Returns:
        decoded_dirs: (n_trials,) decoded directions in radians [0, 2π)
    """
    ### YOUR CODE HERE ###
    pass

# Apply to all 480 trials using the TRUE PDs (neuron_pds)
### YOUR CODE HERE ###

### Exercise 4.2: Evaluate population vector accuracy

The lecture (Figure 8) reported 96% overall, 100% healthy, 92% impaired.

For each trial, find the **nearest** of the 8 target directions to the decoded direction. Compute accuracy overall and split by group. Also plot a histogram of angular errors, split by healthy/impaired.

**Question:** Why does the population vector work worse for impaired subjects?

### Exercise 4.1b: Polar plot of decoded directions (Lecture Figure 8a)

Plot the decoded directions for **healthy subjects** on a polar axis. Each trial is a dot; the 8 true targets are marked as orange circles. The dots should cluster tightly around each target.

In [ ]:
# Exercise 4.1b: Polar plot of decoded directions (healthy only)

### YOUR CODE HERE ###

In [ ]:
# Exercise 4.2: Population vector accuracy and error distribution
# nearest target = argmin of circular distance to each of 8 target_angles

### YOUR CODE HERE ###

### 🤔 Thought Exercise

The lecture (§5) emphasized that the population vector has **zero tunable parameters** — yet it achieves 96%. The lecture's explanation: *"we are not learning a mapping from data — we are inverting the encoding model analytically."*

In your own words: why does this work so well? And why can't it classify healthy vs impaired?

*Write your answer below.*

In [ ]:
# Your reflection:
# 

---

## 🟡 Part 5: Logistic Regression — Same Algorithm, Different Data (Lecture §5)

The lecture isolated the effect of **representation** by running the exact same logistic regression (C = 10, LOSO) on two different inputs: 6 EMG amplitudes vs 80 neural firing rates.

Two tasks: **8-direction decoding** and **binary healthy-vs-impaired**.

### Exercise 5.1: 8-Direction decoding — EMG vs Neural

Run `LogisticRegression(C=10)` with `StandardScaler` in a pipeline, using `LeaveOneGroupOut` on subjects. Compare EMG and neural inputs.

**Expected (from lecture Figure 9):** EMG ~81%, Neural ~91%.

In [ ]:
# Exercise 5.1: 8-Direction decoding comparison
logo = LeaveOneGroupOut()

# Build two pipelines: pipe_emg, pipe_neural
# Run cross_val_score for each on 'targets'

### YOUR CODE HERE ###

### Exercise 5.2: Binary classification — EMG vs Neural

Same pipelines, but now predict `group_binary` (healthy=0, impaired=1).

**Expected (from lecture Figure 9):** EMG ~85%, Neural ~73%. Muscles win here.

In [ ]:
# Exercise 5.2: Binary classification comparison
group_binary = (labels == 'impaired').astype(int)

### YOUR CODE HERE ###

### Exercise 5.3: ROC curves — EMG vs Neural

The lecture (Figure 10e) showed EMG AUC = 0.94, Neural AUC = 0.80. Reproduce this using `cross_val_predict` with `method='predict_proba'`.

Plot both ROC curves on the same axes.

In [ ]:
# Exercise 5.3: ROC curves for the binary task
# Use cross_val_predict(..., method='predict_proba')[:, 1]
# to get LOSO predicted probabilities for each feature set

### YOUR CODE HERE ###

---

## 🔴 Part 6: The Clinical Question — Where Does the Signal Live? (Lecture §6)

The lecture traced the impairment (synergy merging) through the motor pathway. It acts at the **spinal level** — between cortex and muscles. So muscles feel the **direct** effect while cortex shows only **secondary** changes.

The lecture walked through this evidence in three steps: one neuron → all 80 neurons → 6 muscles.

### Exercise 6.0: One neuron — the subtle difference (Lecture Figure 10a)

The lecture traces the impairment step by step: *"Start at a single neuron... the difference is subtle... error bars nearly overlap."*

Plot Neuron 21's tuning curve for healthy (blue) and impaired (red) with mean ± SEM. This reproduces Figure 10a.

In [ ]:
# Exercise 6.0: Single neuron healthy vs impaired (Lecture Figure 10a)

### YOUR CODE HERE ###

### Exercise 6.1: The impairment at the neural level

Reproduce lecture Figure 10b: for all 80 neurons, compute **modulation depth** (max mean rate − min mean rate across 8 directions) separately for healthy and impaired. Scatter plot: healthy on x, impaired on y, diagonal = no change.

**Expected:** Most neurons fall below the diagonal (72/80 per the lecture).

In [ ]:
# Exercise 6.1: Modulation depth scatter — healthy vs impaired
# For each neuron:
#   h_rates = [mean rate at each direction for healthy]
#   depth_h = max(h_rates) - min(h_rates)
#   Same for impaired

### YOUR CODE HERE ###

### Exercise 6.2: The impairment at the muscle level

Reproduce lecture Figure 10c: grouped bar chart of mean EMG amplitude for each of the 6 muscles, healthy and impaired side by side.

**Compare** the visual effect size to the neural scatter plot above. Which level shows more obvious differences?

In [ ]:
# Exercise 6.2: EMG amplitude comparison — healthy vs impaired

### YOUR CODE HERE ###

### 🤔 Thought Exercise

The lecture states: *"No algorithm can extract a signal that the data do not contain."*

A colleague argues that we should use a more powerful model (e.g., deep neural network) to extract the diagnostic signal from neural rates. Based on what you've seen about **where** the impairment acts in the motor pathway, would a fancier algorithm help? Why or why not?

*Write your answer below.*

In [ ]:
# Your reflection:
# 

---

## 🔴 Part 7: Bringing It Together (Lecture §7)

The lecture identified three lessons from the comparison table:
1. **Representation** changed the answer more than any algorithm (81% → 91% by swapping EMG for neural)
2. A **model-free** decoder (population vector, 96%) outperformed a trained classifier (91%)
3. The **best method depends on the task** — neurons for direction, muscles for diagnosis

### Exercise 7.1: Fill in the comparison table

Print all the numbers to complete this table:

| Week | Method | Features | 8-Dir Accuracy | Binary Accuracy | AUC |
|------|--------|----------|---------------|-----------------|-----|
| 5 | Logistic Reg (C=10) | Raw EMG (6) | 80.8% | 85.0% | 0.94 |
| 6 | Logistic Reg (C=10) | Neural (80) | ___% | ___% | ___ |
| 6 | Population Vector | Neural (80) | ___% | N/A | N/A |

The population vector can only decode direction — it cannot classify healthy vs impaired.

In [ ]:
# Exercise 7.1: Print all comparison table numbers

### YOUR CODE HERE ###

### Exercise 7.2: Visualize the double dissociation

Create a grouped bar chart (lecture Figure 10f) with two groups (Direction Decoding, Clinical Diagnosis) and two bars each (EMG, Neural). Mark the winner in each task.

**This is the key result of Week 6:** the best recording site depends on the question.

In [ ]:
# Exercise 7.2: Double dissociation bar chart

### YOUR CODE HERE ###

---

## Summary

This lab followed the 7 sections of the Week 6 lecture:

1. **§1–2:** Explored the neural population — cosine tuning curves and population heatmaps
2. **§3:** Saw why linear/logistic regression fail for spike counts; visualized the Poisson distribution and MLE
3. **§4:** Fit Poisson GLMs — recovered each neuron's PD (median error ~1–2°)
4. **§5a:** Decoded with the population vector — 96% accuracy with zero parameters
5. **§5b:** Compared neural vs muscle decoding — neural wins direction (91% vs 81%), EMG wins diagnosis (85% vs 73%)
6. **§6:** Investigated why — the impairment acts between cortex and muscle, so muscles carry the diagnostic signal
7. **§7:** Updated the comparison table and identified the double dissociation

**Key takeaway:** Before reaching for a fancier algorithm, ask whether you are measuring the right thing.